In [ ]:
# STEP 1: IMPORT LIBRARIES

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

import joblib

In [ ]:
# STEP 2: LOAD DATASET

df = pd.read_csv(r"/content/us_perm_visas.csv", low_memory=False)

print("Original Shape:", df.shape)

Original Shape: (79808, 154)


In [ ]:
# STEP 3: CLEANING
df["country_of_citzenship"] = df["country_of_citzenship"].fillna("Unknown")

df.rename(columns={
    "country_of_citzenship": "country",
    "class_of_admission": "visa_type",
    "employer_state": "processing_office"
}, inplace=True)

df["decision_date"] = pd.to_datetime(df["decision_date"], errors="coerce")
df = df.dropna(subset=["decision_date"])

In [ ]:
# STEP 4: CREATE TARGET

np.random.seed(42)

processing_days = np.random.randint(30, 180, len(df))
df["processing_days"] = processing_days

df["application_date"] = df["decision_date"] - pd.to_timedelta(processing_days, unit="D")

df["processing_days"] = (df["decision_date"] - df["application_date"]).dt.days

In [ ]:
# STEP 5: FEATURE ENGINEERING

df["application_month"] = df["application_date"].dt.month

# Season feature
df["season"] = df["application_month"].apply(
    lambda x: "Peak" if x in [1,2,12] else "Off-Peak"
)

# Country avg
country_avg = df.groupby("country")["processing_days"].mean()
df["country_avg"] = df["country"].map(country_avg)

# Visa avg
visa_avg = df.groupby("visa_type")["processing_days"].mean()
df["visa_avg"] = df["visa_type"].map(visa_avg)

# Fill missing
df.fillna("Unknown", inplace=True)

print("Processed Shape:", df.shape)

/tmp/ipykernel_879/1433233102.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Unknown' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna("Unknown", inplace=True)


Processed Shape: (79808, 160)


In [ ]:
# HANDLE MISSING VALUES
categorical_cols = ["country", "visa_type", "processing_office"]
numerical_cols = ["application_month", "country_avg", "visa_avg"]

# Fill categorical
for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

# Ensure numeric + fill
for col in numerical_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col].fillna(df[col].mean(), inplace=True)

/tmp/ipykernel_879/3659629711.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mean(), inplace=True)


In [ ]:
# STEP 6: FEATURES & TARGET

features = [
    "country",
    "visa_type",
    "processing_office",
    "application_month",
    "country_avg",
    "visa_avg"
]

X = df[features]
y = df["processing_days"]

In [ ]:
# STEP 7: TRAIN TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# STEP 8: PIPELINE
categorical_cols = ["country", "visa_type", "processing_office"]
numerical_cols = ["application_month", "country_avg", "visa_avg"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numerical_cols)
    ]
)

model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_depth=20,
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
# STEP 9: TRAIN MODEL

model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['country', 'visa_type',
                                                   'processing_office']),
                                                 ('num', 'passthrough',
                                                  ['application_month',
                                                   'country_avg',
                                                   'visa_avg'])])),
                ('model',
                 RandomForestRegressor(max_depth=20, n_estimators=200,
                                       n_jobs=-1, random_state=42))])

In [ ]:
# STEP 10: EVALUATION

preds = model_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print("\n FINAL MODEL PERFORMANCE")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)


 FINAL MODEL PERFORMANCE
MAE: 34.59764369780884
RMSE: 40.929983224229474
R2 Score: 0.10204251111357776


In [ ]:
# STEP 11: SAVE MODEL
joblib.dump(model_pipeline, "final_model.pkl")

print("\n Model saved as final_model.pkl")


 Model saved as final_model.pkl


In [ ]:
# STEP 12: TEST WITH SAMPLE INPUT
sample_input = pd.DataFrame([{
    "country": "INDIA",
    "visa_type": "H1B",
    "processing_office": "CA",
    "application_month": 5,
    "country_avg": df["country_avg"].mean(),
    "visa_avg": df["visa_avg"].mean()
}])

prediction = model_pipeline.predict(sample_input)[0]
print("\n Sample Prediction:", int(prediction), "days")


 Sample Prediction: 159 days
